# 输出策略

三种结构化输出策略：ToolStrategy、ProviderStrategy、AutoStrategy。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

| 策略 | 原理 | 速度 |
| --- | --- | --- |
| ToolStrategy | Schema 伪装成工具 | 较慢 |
| ProviderStrategy | 模型原生能力 | 较快 |
| AutoStrategy | 自动检测选择最佳 | 最优 |

## ToolStrategy——工具调用模式

兼容性最好，所有 function calling 模型可用。

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model

class WeatherReport(BaseModel):
    city: str = Field(description="城市")
    temperature: float = Field(description="温度")
    condition: str = Field(description="状况")
    humidity: int = Field(description="湿度")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, response_format=ToolStrategy(schema=WeatherReport), system_prompt="生成结构化天气报告。")
print("ToolStrategy Agent 已创建")

### handle_errors——错误重试

In [ ]:
strategy_with_retry = ToolStrategy(schema=WeatherReport, handle_errors=True)
print(f"带重试的策略: {type(strategy_with_retry).__name__}")

## ProviderStrategy——原生结构化输出

需模型支持（GPT-4o+、Claude 3+）。

In [ ]:
from langchain.agents.structured_output import ProviderStrategy

class CourseInfo(BaseModel):
    name: str = Field(description="课程名称")
    level: str = Field(description="难度")
    price: str = Field(description="价格")

agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0), response_format=ProviderStrategy(schema=CourseInfo), system_prompt="你是课程助手。")
print("ProviderStrategy Agent 已创建")

## AutoStrategy——自动选择（推荐）

直接传 Pydantic 模型即可。

In [ ]:
class Analysis(BaseModel):
    summary: str = Field(description="总结")
    score: int = Field(description="评分 1~10")
    pros: list[str] = Field(description="优点")
    cons: list[str] = Field(description="缺点")

agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0), response_format=Analysis, system_prompt="你是课程评估专家。")
print("AutoStrategy Agent 已创建")

**术语：** ToolStrategy、ProviderStrategy、AutoStrategy、handle_errors